### Setup

In [20]:
import openai
import json
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path("..") / ".env")

True

### Pydantic and Structured Output

In [21]:
from pydantic import BaseModel, Field, ValidationError
from openai import OpenAI, OpenAIError
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type
import logging

In [22]:
# 1. Configure Logging for Observability
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [23]:
# 2. Define the Schema as a Pydantic Model
# The descriptions here act as targeted prompt instructions for the LLM.
class ContactInfo(BaseModel):
    name: str = Field(description="The full name of the individual.")
    company: str = Field(description="The company name. Use 'Unknown' if not present.")
    email: str = Field(description="The email address.")
    phone: str = Field(description="The phone number. Use 'Unknown' if not present.")

In [24]:
# 3. Securely Initialize the Client
# Never hardcode API keys. Always pull from the environment or a secrets manager.
client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [25]:
# 4. Implement Resiliency with Tenacity
# Network calls fail. Rate limits happen. We use exponential backoff for dependability.
@retry(
    wait=wait_exponential(multiplier=1, min=2, max=10), 
    stop=stop_after_attempt(3),
    retry=retry_if_exception_type((OpenAIError, ConnectionError))
)
def extract_contact_info(email_text: str) -> ContactInfo | None:
    """
    Extracts structured contact information from raw email text.
    """
    try:
        # We use the beta.chat.completions.parse endpoint for strict Structured Outputs
        completion = client.beta.chat.completions.parse(
            model="gpt-4o-mini", # The latest fast/efficient model for data extraction
            messages=[
                {
                    "role": "system", 
                    "content": "You are a precise data extraction microservice. Extract the requested information accurately."
                },
                {
                    "role": "user", 
                    # Isolate user data from instructions to mitigate prompt injection
                    "content": f"Extract contact info from this email payload:\n<email>\n{email_text}\n</email>"
                }
            ],
            response_format=ContactInfo, # Pass the Pydantic model directly
            temperature=0.0 # Zero temperature ensures deterministic, repeatable outputs
        )
        
        # The SDK automatically parses the JSON into the Pydantic object
        parsed_data = completion.choices[0].message.parsed
        
        # In rare cases of complete structural refusal, parsed might be None
        if parsed_data is None:
             logger.error("Model refused to output the expected structure.")
             return None
             
        return parsed_data

    except ValidationError as e:
        logger.error(f"Data validation failed (Schema mismatch): {e}")
        raise
    except OpenAIError as e:
        logger.error(f"OpenAI API Error: {e}")
        raise

In [26]:
# 5. Trial run with example email
email_text = """
Hi, I'm Sarah Connor from Cyberdyne Systems. You can reach me at sarah.connor@cyberdynesystems.com
or call me at (555) 121-1210. Looking forward to discussing the skynet plan.
"""

data = extract_contact_info(email_text)
print(data)

2026-04-07 21:03:39,168 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


name='Sarah Connor' company='Cyberdyne Systems' email='sarah.connor@cyberdynesystems.com' phone='(555) 121-1210'
